# Classical Machine Learning Baseline for Patient Risk Prediction

# ----------------------------
# 0. Import packages here
# ----------------------------

In [ ]:
# import the needed packages
import pandas as pd
from sklearn.datasets import make_classification 
# make_classification is used to generate a random n-class classification problem
# This in simple terms means that we can generate a random dataset with a specified number of classes, features, and samples.
# This is great for generating synthetic datasets to test machine learning algorithms.


from sklearn.model_selection import train_test_split
# train_test_split is used to split the dataset into training and testing sets. 
# This is important because we want to train our model on one set of data and test it on another set of data 
# We do this to evaluate model performance on data that was not seen by the model during training.

from sklearn.tree import DecisionTreeClassifier
# DecisionTreeClassifier is a machine learning algorithm that can be used for both classification and regression tasks.
# It works by iteratively breaking the dataset into smaller pieces based on relationships between the features and the target variable.

from sklearn.ensemble import RandomForestClassifier
# Random Forest models work like democracy applied to Decision Trees. 
# Each tree in the forest makes a prediction, and the class with the most votes in the end becomes the model's prediction.
# This leads it to be referred to as "ensemble learning" where multiple models are used to make a prediction.

from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
# classification_report is used to evaluate the performance of a classification model by providing metrics such as precision, recall, and F1-score for each class.
# confusion_matrix is used to evaluate the performance of a classification model by providing a matrix that shows the number of true positives, true negatives, false positives, and false negatives for each class.
# accuracy_score is used to evaluate the performance by providing the overall accuracy of the model, which is the number of correct predictions divided by the total number of predictions.

import matplotlib.pyplot as plt
# Matplotlib is a data visualization library that provides a wide range of plotting capabilities.

import seaborn as sns
# Seaborn is a data visualization library based on matplotlib that provides very nice graphical representations of data. 

# ----------------------------
# 1. Create synthetic patient-like data
# ----------------------------

In [ ]:
X,y= make_classification(
    n_samples=20000, # This is the number of samples we want to generate for the dummy data.
    n_features=12, # This is the number of features we want to generate for the dummy data.
    n_informative=6, # This is the number of "informative features" we want to generate for the dummy data. 
    # Informative features are basically those that are actually useful for predicting the target variable.
    n_redundant=2, # This is the number of redundant features we want to generate for the dummy data. 
    # Redundant features are those that are highly correlated with other features, and hence hold no predictive value.
    n_classes=5, # This is the number of "classes" aka unique groups we want in the dummy data.
    random_state=42 # This is the random state for reproducibility, i.e. if you run this code multiple times, you will get the same results each time.
)

feature_names = [
    "age", "heart_rate", "systolic_bp", "diastolic_bp",
    "spo2", "resp_rate", "glucose", "lactate",
    "creatinine", "hemoglobin", "wbc", "platelets"
]
# feature_names tells us the names of the features in the dataset.

df = pd.DataFrame(X, columns=feature_names) # This just stores our data in a data format.
df["disease_risk"]=y #...while this adds the "classification" column to that dataframe.

classes = df["disease_risk"].unique() # This just gets the unique classes in the dataset.

In [ ]:
df.head(10) # This just prints the first couple rows of the dataframe so we can an idea what madness we just created.

# ----------------------------
# 2. Re-manage & Split data
# ----------------------------

In [ ]:
X = df.drop(columns=["disease_risk"]) # Overwriting the X variable to only contain the features of the dataset.
y = df["disease_risk"] # Overwriting the y variable to only contain the target variable of the dataset.


#Split the data into train and testing sets.
X_train, X_test, y_train, y_test = train_test_split(X,y,
    test_size=0.2, # This is the proportion of the dataset that we want to use for testing.
    random_state=42, # This is the random state for reproducibility.
    stratify=y # This is used to ensure that the distribution of classes in the training and testing sets is similar to that of the original dataset.
    # This ensures that the model is trained on a representative sample distribution.
)

# ----------------------------
# 3. Train Decision Tree
# ----------------------------

In [ ]:
dt_model = DecisionTreeClassifier(
    max_depth=10, # This is the maximum depth of the decision tree.
    random_state=42 # This is the random state for reproducibility.
    # NOTE: The documentation contains a bunch of other hyperparameters that we can tune.
)

dt_model.fit(X_train, y_train) # This is where the model is actually trained on the training data.

In [ ]:
dt_predictions = dt_model.predict(X_test) # This is where the model is used to make predictions on the testing data.
dt_accuracy = accuracy_score(y_test, dt_predictions) # This is where the accuracy of the model is calculated.

### Decision Tree Results

In [ ]:
print("Accuracy:", dt_accuracy)

In [ ]:
def plot_confusion_matrix(y_test, dt_predictions, classes):
    cm= confusion_matrix(y_test, dt_predictions)
    df_cm = pd.DataFrame(cm, index=classes, columns=classes)

    # 3. Plot with Seaborn
    plt.figure(figsize=(4, 3)) # Set the figure size
    sns.set_theme(style="white") # Set a clean theme

    # 'annot=True' shows the numbers, 'fmt="d"' ensures they are integers
    # 'cmap' can be 'Blues', 'viridis', 'magma', 'rocket', etc.
    sns.heatmap(df_cm,
                annot=True, # This shows the numbers in the heatmap.
                fmt="d", # Alternatively, you can use "f" for floating point numbers.
                cmap="coolwarm", # Alternatively, you can use  "viridis", "magma", etc.
                cbar=True, 
                linewidths=1.5, 
                annot_kws={
                    "size": 16,
                    "weight": "bold"
                    } # This sets the size and weight of the annotations in the heatmap.
                )

    plt.title('Confusion Matrix', fontsize=18, pad=20)
    plt.ylabel('Actual Label', fontsize=14)
    plt.xlabel('Predicted Label', fontsize=14)
    plt.show()

In [ ]:
plot_confusion_matrix(y_test, dt_predictions, classes) # This is where the confusion matrix is plotted.

In [ ]:
def plot_classification_report(y_test, dt_predictions):
    report_dict=classification_report(y_test,
                                       dt_predictions,
                                       output_dict=True # This makes the output a dictionary instead of a string.
                                       )

    df_report = pd.DataFrame(report_dict).transpose()
    # This just converts the classification report into a dataframe for easier viewing.

    # Apply styling (e.g., color gradient for scores)
    styled_report = df_report.style.background_gradient(
        cmap='RdYlGn',  # Alternatively, you can use 'coolwarm', 'viridis', etc.
        subset=pd.IndexSlice[df_report.index[:-3], ['precision', 'recall', 'f1-score']]
    )
    display(styled_report)


In [ ]:
plot_classification_report(y_test, dt_predictions) # This is where the classification report is plotted.

In [ ]:
importance_df = pd.DataFrame({
    "feature": feature_names,
    "importance": dt_model.feature_importances_,
}).sort_values(by="importance", ascending=False)

print("\nDT Feature Importance")
print(importance_df)

# ----------------------------
# 4. Train Random Forest
# ----------------------------

In [ ]:
rf_model = RandomForestClassifier(
    n_estimators=200, # This is the number of trees in the forest. More trees usually lead to better performance, but also increase computation time.
    max_depth=8, # This is the maximum depth of each tree. Deeper trees can capture more complex patterns but may lead to overfitting.
    random_state=42 # This ensures reproducibility of the results.
)

rf_model.fit(X_train, y_train) # This is where the model is actually trained on the training data.

In [ ]:
rf_predictions = rf_model.predict(X_test) # This is where the model is used to make predictions on the testing data.
rf_accuracy = accuracy_score(y_test, rf_predictions) # This is where the accuracy of the model is calculated.

### Random Forest Results

In [ ]:
print("Accuracy:", rf_accuracy)

In [ ]:
plot_confusion_matrix(y_test, rf_predictions, classes) # This is where the confusion matrix is plotted.

In [ ]:
plot_classification_report(y_test, rf_predictions) # This is where the classification report is plotted.

# ----------------------------
# 5. Feature importance
# ----------------------------

In [ ]:
importance_df = pd.DataFrame({
    "feature": feature_names,
    "importance": rf_model.feature_importances_,
}).sort_values(by="importance", ascending=False)

print("\nRF Feature Importance")
print(importance_df)

# ----------------------------
# 6. Save outputs
# ----------------------------

In [ ]:
importance_df.to_csv("results/feature_importance.csv", index=False)

with open("results/model_summary.txt", "w") as f:
    f.write("Classical ML Baseline for Patient Risk Prediction\n")
    f.write("Models compared: Decision Tree and Random Forest\n")
    f.write(f"Decision Tree Accuracy: {dt_accuracy:.4f}\n")
    f.write(f"Random Forest Accuracy: {rf_accuracy:.4f}\n")

print("\nSaved results to the results/ folder.")